# Solar_K final pipeline, end to end

Reproduces the frozen Spec 06 pipeline (`E007tta4s` + `E015tta4s`, the TTA-regenerated probability maps
from E023) on one CV fold of the real P5 protocol, using only `src/solar_k` and pre-computed
probability maps present under `outputs/`. **No GPU, no network, and no re-training** -- this is
the same `src/solar_k.cv_pipeline.run_p5` that every experiment in `experiments.csv` was scored with,
called directly instead of through `scripts/run_p5.py`, restricted to CV fold 0 (~140 held-out images)
so it finishes in a few minutes on a laptop CPU.

See `reports/06_final_report.md` for the full 5-fold numbers and `README.md` "Reproduce from scratch"
for the GPU-requiring steps (training, TTA inference) this notebook skips by using the already-generated, not-committed
OOF maps.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from solar_k import cv_pipeline as cvp, ensemble as skens, io as skio

In [ ]:
folds_df = pd.read_csv(skio.data_root() / "splits" / "folds_v1.csv")
day_by_stem = dict(zip(folds_df["file_stem"], folds_df["date"]))
site_by_stem = dict(zip(folds_df["file_stem"], folds_df["site"]))
fold_by_stem = dict(zip(folds_df["file_stem"], folds_df["fold"].astype(int)))

members = ["E007tta4s", "E015tta4s"]  # the frozen pipeline's members (Spec 06 E023/E024)
stems = skens.available_oof_stems(members, fold_by_stem)
print(f"{len(stems)} images with OOF maps for both members")  # 707 if the full OOF (outputs/, gitignored) is present

In [ ]:
import solar_k.preprocess as skpre

train_paths = {p.stem: p for p in skio.iter_train_images()}
disk_info_by_stem = {s: skpre.load_or_build(s, train_paths[s])[1] for s in stems}
instance_stats = pd.read_csv(skio.data_root() / "interim" / "instance_stats.csv")
quartile_edges = instance_stats["area"].quantile([0.25, 0.5, 0.75]).tolist()
quartile_edges

## Run one CV fold of P5

This fits cfg1 / shape-gate+scorer / regrow on that fold's day-grouped train portion and evaluates on
the held-out portion -- the exact protocol `reports/05_ensemble_report.md` §2-3 and
`reports/06_final_report.md` describe. `workers=0` runs in-process (no multiprocessing spawn needed
for a notebook kernel).

In [ ]:
res = cvp.run_p5(
    members, stems, day_by_stem, site_by_stem, disk_info_by_stem, quartile_edges,
    workers=0, cv_folds=[0], log=print,
)
fold0 = res.per_fold[0]
print(f"CV fold 0: PQ={fold0['pq']:.4f} SQ={fold0['sq']:.4f} RQ={fold0['rq']:.4f} "
      f"TP={fold0['tp']} FP={fold0['fp']} FN={fold0['fn']} on {fold0['n_held']} held-out images")

The full 5-fold mean in `reports/06_final_report.md` is **PQ 0.4302 ± 0.0052**; one fold alone is
noisier (see the std) but should land in the same range.

## Visualize one held-out prediction vs ground truth

In [ ]:
import solar_k.postprocess as skpp

held = res.per_image[res.per_image["cv_fold"] == 0]["stem"].tolist()
stem = held[0]
cfg = skpp.PostprocessConfig.from_dict(fold0["cfg1"])
probs = skens.load_oof_mean(members, stem, fold_by_stem)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(probs, cmap="gray")
ax[0].set_title(f"{stem}: mean probability map")
ax[1].imshow(probs > cfg.t_low, cmap="gray")
ax[1].set_title(f"thresholded at t_low={cfg.t_low}")
for a in ax:
    a.axis("off")
plt.tight_layout()

`reports/figures/06_overlay_gallery.png` has the curated typical / worst-split / worst-merge /
recovered-faint-filament examples with full instance-level overlays against ground truth, generated
on the RunPod GPU box where the 5-fold masks live (`scripts/make_gallery.py`).